In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../src")
from utils import *

In [ ]:
class Counter4Bit:
    """A Ripple Counter that counts"""
    def __init__(self,):
        self.nbits = 4
        self.flipflops = [EdgeTriggeredDTypeFlipFlopWithPresetAndClear() for _ in range(self.nbits)]
        self.and_gate = AND()

    def getQs(self):
        qs = [ff.getQ() for ff in self.flipflops]
        return list(reversed(qs))
    
    def SetMaxAddr(self):
        for ff in self.flipflops:
            ff([0, 0, 1, 0]) # D = 0, CLK = 0, PRE = 1, CLR = 0

    def Clear(self):
        for ff in self.flipflops:
            ff([0, 0, 0, 1]) # D = 0, CLK = 0, PRE = 0, CLR = 1

    def __call__(self, clk = 0, clear_wire = 0):
        # The hardware stabilization loop
        while True:
            # 1. Take a snapshot of the current state of the ff
            old_qs = [ff.getQ() for ff in self.flipflops]

            # 3. Propagate the clock and data through the ripple chain
            current_clk = clk
            for ff in self.flipflops:
                data = ff.getQ_bar()
                # Pass Data, Clock, Preset(0), and clear_p370 wire (0)
                q, q_bar = ff([data, current_clk, 0, clear_wire])
                current_clk = q_bar

            # 4. Read the new state
            new_qs = [ff.getQ() for ff in self.flipflops]

            # 5. If the electrons have settled and nothing changed, break!
            if old_qs == new_qs:
                break

        # Return with MSB on the left for easy reading
        return list(reversed(new_qs))

In [14]:
def test_counter4Bit():
    my_oscillator = Oscillator()
    nbits = 4
    my_counter = Counter4Bit()
    
    previous_clock_state = -1
    golden_clock_state = 0
    golden_qs = [0] * nbits
    golden_value = 0
    for clk in range(64):
        assert golden_clock_state == my_oscillator.level(), f"At clock {clk}, expected clock state {golden_clock_state} but got {my_oscillator.level()}"
        my_counter(my_oscillator.level())
        if previous_clock_state == 0 and golden_clock_state == 1:  # Only increment the expected counter value on the rising edge of the clock
            golden_value = (golden_value + 1) % (2 ** nbits)  # Increment and wrap around at 2^nbits
            golden_qs = int_to_nbit_list(golden_value, nbits)

        if clk < 40:
            print(f"At clock {clk}, expected counter value {golden_value} but got {int(''.join(str(bit) for bit in my_counter.getQs()))}")
        assert golden_qs == my_counter.getQs(), f"At clock {clk}, expected counter value {golden_value} but got {int(''.join(str(bit) for bit in my_counter.getQs()))}"
        previous_clock_state = golden_clock_state
        my_oscillator.tick()  # Move to the next clock state for the next iteration
        golden_clock_state = 1 - golden_clock_state  # Toggle the expected clock state for

    my_counter.Clear()
    print(f"At clock {clk}, expected counter value {golden_value} but got {int(''.join(str(bit) for bit in my_counter.getQs()))}")

test_counter4Bit()

At clock 0, expected counter value 0 but got 0
At clock 1, expected counter value 1 but got 1
At clock 2, expected counter value 1 but got 1
At clock 3, expected counter value 2 but got 10
At clock 4, expected counter value 2 but got 10
At clock 5, expected counter value 3 but got 11
At clock 6, expected counter value 3 but got 11
At clock 7, expected counter value 4 but got 100
At clock 8, expected counter value 4 but got 100
At clock 9, expected counter value 5 but got 101
At clock 10, expected counter value 5 but got 101
At clock 11, expected counter value 6 but got 110
At clock 12, expected counter value 6 but got 110
At clock 13, expected counter value 7 but got 111
At clock 14, expected counter value 7 but got 111
At clock 15, expected counter value 8 but got 1000
At clock 16, expected counter value 8 but got 1000
At clock 17, expected counter value 9 but got 1001
At clock 18, expected counter value 9 but got 1001
At clock 19, expected counter value 10 but got 1010
At clock 20, e

In [29]:
x = [1, 3, 4, 2, 2]
print(x[2:])

[4, 2, 2]


In [43]:
import sys
sys.path.append("../src")
from utils import Decoder_2_4_V2

def test_decoder_2_4():
    print("Testing Decoder_2_4_V2...")
    decoder = Decoder_2_4_V2()

    # ==========================================
    # 1. Test Address Decoding Logic
    # ==========================================
    print("  -> Testing Address Decoding...")
    
    # Input 00 -> Output 0 is High
    assert decoder([0, 0]) == [1, 0, 0, 0], "Decoder failed on input [0, 0]"
    
    # Input 01 -> Output 1 is High
    assert decoder([0, 1]) == [0, 1, 0, 0], "Decoder failed on input [0, 1]"
    
    # Input 10 -> Output 2 is High
    assert decoder([1, 0]) == [0, 0, 1, 0], "Decoder failed on input [1, 0]"
    
    # Input 11 -> Output 3 is High
    assert decoder([1, 1]) == [0, 0, 0, 1], "Decoder failed on input [1, 1]"


    # ==========================================
    # 2. Test Write Signal Routing
    # ==========================================
    print("  -> Testing Write Masking...")
    test_decoded_address = [0, 1, 0, 0] # Pretend address [0, 1] was selected

    # When Write = 1, the selected line should pass through
    write_enabled = decoder.write(test_decoded_address, write=1)
    assert write_enabled == [0, 1, 0, 0], "Write=1 blocked the signal!"

    # When Write = 0, the AND gates should block all signals
    write_disabled = decoder.write(test_decoded_address, write=0)
    assert write_disabled == [0, 0, 0, 0], "Write=0 leaked the signal!"

    print("Decoder_2_4_V2 PASSED!\n")

if __name__ == "__main__":
    test_decoder_2_4()

Testing Decoder_2_4_V2...
  -> Testing Address Decoding...
  -> Testing Write Masking...
Decoder_2_4_V2 PASSED!



In [ ]:
class ControlSignal:
    def __init__(self):
        self.nbits_of_latch = 8

        # TODO(PengChen:) V2 is ambiguous
        self.decoder_2_4_of_latch_1 = Decoder_2_4_V2()
        self.decoder_3_8_1_of_latch_1 = Decoder_3_8()
        self.decoder_3_8_2_of_latch_1 = Decoder_3_8()

        # TODO(PengChen:) for just for simple 
        self.and_gate_2_in = AND()
        self.and_gate_3_in = AND(3)

        self.invert_gate = Inverter()

        self.or_gate_3_in = OR(3)
        self.or_gate_2_in = OR()

        self.my_counter = Counter4Bit()
        self.decoder = Decoder_4_16()
        self.or_gate = OR()

        self.tri_buffer = TriStateBuffer(1)
        self.tri_buffer_2_in = TriStateBuffer(2)
        self.tri_buffer_1_in = TriStateBuffer(1)
        self.tri_buffer_3_in = TriStateBuffer(3)
        
        self.bus1_in_addr_bus_execute = DataBus(num_buffers=6, nbits=1)
        self.bus2_in_addr_bus_execute = DataBus(num_buffers=2, nbits=1)

        self.bus1_in_data_bus_execute = DataBus(num_buffers=3, nbits=1)

        self.tri_buffer_4_in = TriStateBuffer(4)

    def __call__(self, cycle_clk, pulse, reset, latch1):
        self.my_counter(cycle_clk)
        output_of_decoder = self.decoder(self.my_counter.getQs())
        assert len(latch1) == self.nbits_of_latch
# page 366
# --------------------------------------------------------------------------------------------------
        output_of_c7_c6_p366 = self.decoder_2_4_of_latch_1(latch1[:2])
        output_of_c5_c4_c3_p366 = self.decoder_3_8_1_of_latch_1(latch1[2:5])
        output_of_c2_c1_c0_p366 = self.decoder_3_8_2_of_latch_1(latch1[5:])

        move_group_p366 = output_of_c7_c6_p366[1]
        arithmetic_logic_group_p366 = output_of_c7_c6_p366[2]
        memory_source_p366 = output_of_c2_c1_c0_p366[6]
        memory_destination_p366 = output_of_c5_c4_c3_p366[6]

        move_immediates_p366 = self.and_gate_2_in([output_of_c2_c1_c0_p366[6], output_of_c7_c6_p366[0]])
        adi_data_p366 = self.and_gate_2_in([output_of_c2_c1_c0_p366[6], output_of_c7_c6_p366[3]])

        inx_hl_p366 = self.and_gate_3_in([output_of_c2_c1_c0_p366[3], output_of_c5_c4_c3_p366[4], output_of_c7_c6_p366[0]])
        dcx_hl_p366 = self.and_gate_3_in([output_of_c2_c1_c0_p366[3], output_of_c5_c4_c3_p366[5], output_of_c7_c6_p366[0]])

        lda_p366 = self.and_gate_3_in([output_of_c2_c1_c0_p366[2], output_of_c5_c4_c3_p366[7], output_of_c7_c6_p366[0]])
        sta_p366 = self.and_gate_3_in([output_of_c2_c1_c0_p366[2], output_of_c5_c4_c3_p366[6], output_of_c7_c6_p366[0]])
# page 367 first part
# --------------------------------------------------------------------------------------------------
        move_r_r_p367 = self.and_gate_3_in([move_group_p366, self.invert_gate([memory_source_p366]), self.invert_gate([memory_destination_p366])])
        move_r_M_p367 = self.and_gate_3_in(move_group_p366, memory_source_p366, self.invert_gate([memory_destination_p366]))
        move_M_r_p367 = self.and_gate_3_in(move_group_p366, self.invert_gate([memory_source_p366]), memory_destination_p366)

        hlt_p367 = self.and_gate_3_in([move_group_p366, memory_source_p366, memory_destination_p366])

        mvi_r_data_p367 = self.and_gate_2_in([move_immediates_p366, self.invert_gate([memory_destination_p366])])
        mvi_M_data_p367 = self.and_gate_2_in([move_immediates_p366, memory_destination_p366])

        add_r_p367 = self.and_gate_2_in([arithmetic_logic_group_p366, self.invert_gate([memory_source_p366])])
        add_M_p367 = self.and_gate_2_in([arithmetic_logic_group_p366, memory_source_p366])
# page 367 second part
# --------------------------------------------------------------------------------------------------
        fetch_2_byte_p367 = self.or_gate_3_in([mvi_r_data_p367, mvi_M_data_p367, adi_data_p366])
        fetch_3_byte_p367 = self.or_gate_2_in([lda_p366, sta_p366])
        fetch_1_byte_invert_p367 = self.or_gate_2_in([fetch_2_byte_p367, fetch_3_byte_p367])
        fetch_1_byte_p367 = self.invert_gate([fetch_1_byte_invert_p367])

        execute_2_cycle_p367 = self.or_gate_3_in([adi_data_p366, add_r_p367, add_M_p367])
        execute_1_cycle_p367 = self.invert_gate([execute_2_cycle_p367])
# page 370
# --------------------------------------------------------------------------------------------------
        fetch_cycle_1_out_p370 = output_of_decoder[0]
        fetch_cycle_2_out_p370 = self.and_gate_2_in([output_of_decoder[2], fetch_1_byte_invert_p367])
        fetch_cycle_3_out_p370 = self.and_gate_2_in([output_of_decoder[4], fetch_3_byte_p367])
        pc_increment_p370 = self.or_gate_3_in([self.and_gate_2_in([output_of_decoder[3], fetch_1_byte_invert_p367]), output_of_decoder[1], self.and_gate_2_in([output_of_decoder[5], fetch_3_byte_p367])])
        # EC1 on page 373
        exec_cycle_1_out_p370 = self.or_gate_3_in([self.and_gate_2_in([output_of_decoder[2], fetch_1_byte_p367]), self.and_gate_2_in([output_of_decoder[4], fetch_2_byte_p367]), self.and_gate_2_in([output_of_decoder[6], fetch_3_byte_p367])])

        tmp_exec_cycle_2_p370 = self.or_gate_3_in([self.and_gate_2_in([output_of_decoder[3], fetch_1_byte_p367]), self.and_gate_2_in([output_of_decoder[5], fetch_2_byte_p367]), self.and_gate_2_in([output_of_decoder[7], fetch_3_byte_p367])])
        # EC2 on page 373
        exec_cycle_2_out_p370 = self.and_gate_2_in([tmp_exec_cycle_2_p370, execute_2_cycle_p367])

        tmp1_reset_p370 = self.or_gate_3_in([self.and_gate_2_in([output_of_decoder[4], fetch_1_byte_p367]), self.and_gate_2_in([output_of_decoder[6], fetch_2_byte_p367]), self.and_gate_2_in([output_of_decoder[8], fetch_3_byte_p367])])
        tmp2_reset_p370 = self.or_gate_2_in([self.and_gate_2_in([tmp_exec_cycle_2_p370, execute_1_cycle_p367]), self.and_gate_2_in([execute_2_cycle_p367, tmp1_reset_p370])])
        clear_p370 = self.or_gate_2_in([reset, tmp2_reset_p370])

        if clear_p370:
            self.my_counter.Clear() # clear_p370 0 and output[0] = 1 -> fetch cycle 1 next cycle
# page 372 first part
# --------------------------------------------------------------------------------------------------
        or_of_fetch_p372 = self.or_gate_3_in([fetch_cycle_1_out_p370, fetch_cycle_2_out_p370, fetch_cycle_3_out_p370])
        program_counter_enable_p372 = or_of_fetch_p372
        # here input and enable signal is same for tri-buffer
        # ATTENTION: maybe merge with other component can enable this.
        ram_data_out_enable_fetch_phrase_p372 = self.tri_buffer(or_of_fetch_p372, or_of_fetch_p372)[0]
        input_of_inc_dec_clk_p372 = self.and_gate_2_in([or_of_fetch_p372, pulse])
        # ATTENTION: maybe merge with other component can enable this.
        inc_dec_clk_fetch_phrase_p372 = self.tri_buffer(input_of_inc_dec_clk_p372, input_of_inc_dec_clk_p372)[0]

        instru_latch_1_clk_p372 = self.and_gate_2_in([fetch_cycle_1_out_p370, pulse])
        instru_latch_2_clk_p372 = self.and_gate_2_in([fetch_cycle_2_out_p370, pulse])
        instru_latch_3_clk_p372 = self.and_gate_2_in([fetch_cycle_3_out_p370, pulse])
# page 372 second part
# --------------------------------------------------------------------------------------------------
        # ATTENTION: maybe merge with other component can enable this.
        increment_enable_fetch_phrase_p372 = self.tri_buffer(pc_increment_p370, pc_increment_p370)[0]
        input_of_program_counter_clk_fetch_phrase_p372 = self.and_gate_2_in([pc_increment_p370, pulse])
        # ATTENTION: maybe merge with other component can enable this.
        program_counter_clk_fetch_phrase_p372 = self.tri_buffer([input_of_program_counter_clk_fetch_phrase_p372, input_of_program_counter_clk_fetch_phrase_p372])[0]

# page 373 first part
# --------------------------------------------------------------------------------------------------
        execute_pulse_1_decode_phrase_p373 = self.and_gate_2_in([exec_cycle_1_out_p370, pulse])
        execute_pulse_2_decode_phrase_p373 = self.and_gate_2_in([exec_cycle_2_out_p370, pulse])
# page 373 second part
# --------------------------------------------------------------------------------------------------
        # this halt output will goes to the circuit with the oscillator
        halt_exec_phrase_p373 = self.and_gate_2_in([hlt_p367, execute_pulse_1_decode_phrase_p373])

# page 374
# --------------------------------------------------------------------------------------------------
# addr bus
        # ATTENTION: maybe merge with other component can enable this.
        hl_enable_addr_bus_exec_phrase_1_p374, inst_latch2_3_enable_addr_bus_exec_phrase_1_p374 = self.tri_buffer_2_in([self.bus1_in_addr_bus_execute([move_r_M_p367, 
                                                                        move_M_r_p367,
                                                                        mvi_M_data_p367,
                                                                        add_M_p367,
                                                                        inx_hl_p366,
                                                                        dcx_hl_p366]), self.bus2_in_addr_bus_execute([lda_p366,
                                                                                                              sta_p366])], exec_cycle_1_out_p370)
        # ATTENTION: maybe merge with other component can enable this.
        inc_dec_clk_enable_addr_bus_exec_phrase_1_p374 = self.tri_buffer_1_in([self.bus2_in_addr_bus_execute([inx_hl_p366, 
                                                                           dcx_hl_p366])], execute_pulse_1_decode_phrase_p373)[0]
        # ATTENTION: maybe merge with other component can enable this.
        hl_select_enable_addr_bus_exec_phrase_2_p374, increment_enable_addr_bus_exec_phrase_2_p374, dec_enable_addr_bus_exec_phrase_2_p374 = self.tri_buffer_3_in([self.bus2_in_addr_bus_execute([inx_hl_p366, 
                                                                           dcx_hl_p366]), inx_hl_p366, dcx_hl_p366], exec_cycle_2_out_p370)
        # ATTENTION: maybe merge with other component can enable this.
        hl_clk_addr_bus_exec_phrase_2_p374 = self.tri_buffer_1_in([self.bus2_in_addr_bus_execute([inx_hl_p366, 
                                                                      dcx_hl_p366])], execute_pulse_2_decode_phrase_p373)[0]
# page 375
# --------------------------------------------------------------------------------------------------
        # ATTENTION: maybe merge with other component can enable this.
        ra_enable_data_bus_exec_phrase_1_p375, ram_data_out_enable_data_bus_exec_phrase_1_p375, inst_latch2_enable_data_bus_exec_phrase_1_p375, acc_enable_data_bus_exec_phrase_1_p375 = self.tri_buffer_4_in([self.bus1_in_data_bus_execute([move_r_r_p367,
                                                                                                                              move_M_r_p367,
                                                                                                                              add_r_p367,
                                                                                                                              ]),
                                                                                                self.bus1_in_data_bus_execute([move_r_M_p367,
                                                                                                                              add_M_p367,
                                                                                                                              lda_p366,
                                                                                                                              ]),
                                                                                                self.bus1_in_data_bus_execute([mvi_r_data_p367,
                                                                                                                              mvi_M_data_p367,
                                                                                                                              adi_data_p366,
                                                                                                                              ]), sta_p366], exec_cycle_1_out_p370)
        # ATTENTION: maybe merge with other component can enable this.
        ra_clk_data_bus_exec_phrase_1_p375, ram_write_enable_data_bus_exec_phrase_1_p375, alu_clk_data_bus_exec_phrase_1_p375, acc_clk_data_bus_exec_phrase_1_p375 = self.tri_buffer_4_in([self.bus1_in_data_bus_execute([move_r_r_p367,
                                                                                                move_r_M_p367,
                                                                                                mvi_r_data_p367,
                                                                                                ]),
                                                                self.bus1_in_data_bus_execute([move_M_r_p367,
                                                                                                mvi_M_data_p367,
                                                                                                sta_p366,
                                                                                                ]),
                                                                self.bus1_in_data_bus_execute([add_r_p367,
                                                                                                add_M_p367,
                                                                                                adi_data_p366,
                                                                                                ]), lda_p366], execute_pulse_1_decode_phrase_p373)
        
# page 376
# --------------------------------------------------------------------------------------------------
        # ATTENTION: maybe merge with other component can enable this.
        alu_enable_data_bus_exec_phrase_2_p376 = self.tri_buffer_1_in([self.bus1_in_data_bus_execute([add_r_p367,
                                                                         add_M_p367,
                                                                         adi_data_p366,
                                                                         ]), exec_cycle_2_out_p370])[0]
        # ATTENTION: maybe merge with other component can enable this.
        acc_clk_data_bus_exec_phrase_2_p376 = self.tri_buffer_1_in([self.bus1_in_data_bus_execute([add_r_p367,
                                                                       add_M_p367,
                                                                       adi_data_p366,
                                                                       ]), execute_pulse_2_decode_phrase_p373])[0]

        # FIX: Combine signals that are triggered in multiple places using an OR gate!
        # (Apply this OR logic to any other wires you calculated twice)
        # this happen on insruct fetch and data in phrase 1
        final_ram_data_out_enable = self.or_gate_2_in([ram_data_out_enable_fetch_phrase_p372, ram_data_out_enable_data_bus_exec_phrase_1_p375])
        # happen on instruct fetch and addr bus exec phrase 1
        final_inc_dec_clk = self.or_gate_2_in([inc_dec_clk_fetch_phrase_p372, inc_dec_clk_enable_addr_bus_exec_phrase_1_p374])
        # happen on instruct fetch and addr bus exec phrase 2
        final_increment_enable = self.or_gate_2_in([increment_enable_fetch_phrase_p372, increment_enable_addr_bus_exec_phrase_2_p374])
        # happen on data bus phrase 1 for LDA and phrase 2 for ADD r, ADD M, and ADI Data
        final_acc_clk = self.or_gate_2_in([acc_clk_data_bus_exec_phrase_1_p375, acc_clk_data_bus_exec_phrase_2_p376])
        # l think maybe l will loss some enable or clock to get final result use or gate because circuit is so messy.
        # l try my best to double check. hope AI can help me further check whether have duplicate enable/ clock to merge get
        # final result.


        # Pack the Control Bus ribbon cable!
        control_bus = {
            "pc_enable": program_counter_enable_p372,
            "pc_clk": program_counter_clk_fetch_phrase_p372,
            "inc_dec_clk": final_inc_dec_clk,
            "inc_enable": final_increment_enable,
            "dec_enable": dec_enable_addr_bus_exec_phrase_2_p374,

            "inst_latch_1_clk": instru_latch_1_clk_p372,
            "inst_latch_2_clk": instru_latch_2_clk_p372,
            "inst_latch_3_clk": instru_latch_3_clk_p372,
            "inst_latch_2_enable": inst_latch2_enable_data_bus_exec_phrase_1_p375,
            "inst_latch2_3_enable": inst_latch2_3_enable_addr_bus_exec_phrase_1_p374,

            "ram_data_out_enable": final_ram_data_out_enable,
            "ram_write_enable": ram_write_enable_data_bus_exec_phrase_1_p375,

            "ra_enable": ra_enable_data_bus_exec_phrase_1_p375,
            "ra_clk": ra_clk_data_bus_exec_phrase_1_p375,
            "hl_enable": hl_enable_addr_bus_exec_phrase_1_p374,
            "hl_select": hl_select_enable_addr_bus_exec_phrase_2_p374,
            "hl_clk": hl_clk_addr_bus_exec_phrase_2_p374,
            "acc_enable": acc_enable_data_bus_exec_phrase_1_p375,
            "acc_clk": final_acc_clk,

            "alu_enable": alu_enable_data_bus_exec_phrase_2_p376,
            "alu_clk": alu_clk_data_bus_exec_phrase_1_p375,

            "halt": halt_exec_phrase_p373,
            "clear": clear_p370
        }
        return control_bus   


In [54]:
class BasicTiming:
    def __init__(self):
        self.edge_flipflop0 = EdgeTriggeredDTypeFlipFlopWithPresetAndClear()
        self.edge_flipflop1 = EdgeTriggeredDTypeFlipFlopWithPresetAndClear()
        self.edge_flipflop2 = EdgeTriggeredDTypeFlipFlopWithPresetAndClear()
        self.invert1 = Inverter()
        self.and_gate = AND()
        self.and_gate0 = AND()
        
        self.cycle_clock = 0
        self.pulse = 0
    
    def read_cycle_clk(self):
        return self.cycle_clock
    
    def read_pulse(self):
        return self.pulse
    
    def __call__(self, clock, reset=0, halt=0):
        if reset == 1:
            self.edge_flipflop0.Reset() # clear_p370 
            self.edge_flipflop1.Reset()
            self.edge_flipflop2.Reset()
            return ;
        
        self.edge_flipflop0([self.edge_flipflop0.getQ_bar(), halt])

        clock = self.and_gate0([self.edge_flipflop0.getQ_bar(), clock])

        not_clk = self.invert1([clock])

        # while True:
        #     old_q1, old_q_bar1 = self.edge_flipflop1.getQ(), self.edge_flipflop1.getQ_bar()
        #     old_q2, old_q_bar2 = self.edge_flipflop2.getQ(), self.edge_flipflop2.getQ_bar()

        self.edge_flipflop1([self.edge_flipflop1.getQ_bar(), clock])
        self.edge_flipflop2([self.edge_flipflop1.getQ(), not_clk])
        clk_input_to_counter = self.edge_flipflop1.getQ()
        pulse = self.and_gate([self.edge_flipflop1.getQ_bar(), self.edge_flipflop2.getQ()])
        self.cycle_clock = clk_input_to_counter
        self.pulse = pulse
        # return [clk_input_to_counter, pulse]

In [55]:
def test_basic_timing():
    print("--- STARTING CPU TIMING SIMULATION ---")
    timing_unit = BasicTiming()
    my_oscillator = Oscillator()
    
    # We will simulate 3 phases: Normal, Halt, and Reset
    reset_signal = 0
    halt_signal = 0
    
    print("PHASE 1: Normal Execution (Notice the Pulse fires every 2nd clock!)")
    print("Tick | Osc_Clock | Cycle_Clock | Pulse")
    print("---------------------------------------")
    
    for tick in range(1, 9):
        osc_clock = my_oscillator.level()
        
        # 1. Feed the signals into the timing unit
        timing_unit(osc_clock, reset_signal, halt_signal)
        
        # 2. Read the outputs
        cycle_clk = timing_unit.read_cycle_clk()
        pulse = timing_unit.read_pulse()
        
        print(f" {tick:2d}  |     {osc_clock}     |      {cycle_clk}      |   {pulse}")
        
        # 3. Advance time
        my_oscillator.tick()

    print("\nPHASE 2: The HALT Signal (Executing 1100 instruction)")
    halt_signal = 1
    for tick in range(9, 13):
        osc_clock = my_oscillator.level()
        timing_unit(osc_clock, reset_signal, halt_signal)
        
        cycle_clk = timing_unit.read_cycle_clk()
        pulse = timing_unit.read_pulse()
        print(f" {tick:2d}  |     {osc_clock}     |      {cycle_clk}      |   {pulse}  <-- Frozen!")
        my_oscillator.tick()
        
        # Drop the halt signal back to 0 so we don't accidentally toggle the flip-flop again
        halt_signal = 0 

    print("\nPHASE 3: The RESET Button (Hard Reboot)")
    reset_signal = 1
    for tick in range(13, 20):
        osc_clock = my_oscillator.level()
        timing_unit(osc_clock, reset_signal, halt_signal)
        
        cycle_clk = timing_unit.read_cycle_clk()
        pulse = timing_unit.read_pulse()
        print(f" {tick:2d}  |     {osc_clock}     |      {cycle_clk}      |   {pulse}  <-- Cleared!")
        my_oscillator.tick()
    reset_signal = 0
    for tick in range(20, 28):
        osc_clock = my_oscillator.level()
        timing_unit(osc_clock, reset_signal, halt_signal)
        
        cycle_clk = timing_unit.read_cycle_clk()
        pulse = timing_unit.read_pulse()
        print(f" {tick:2d}  |     {osc_clock}     |      {cycle_clk}      |   {pulse}")
        my_oscillator.tick()

test_basic_timing()

--- STARTING CPU TIMING SIMULATION ---
PHASE 1: Normal Execution (Notice the Pulse fires every 2nd clock!)
Tick | Osc_Clock | Cycle_Clock | Pulse
---------------------------------------
  1  |     0     |      0      |   0
  2  |     1     |      1      |   0
  3  |     0     |      1      |   0
  4  |     1     |      0      |   1
  5  |     0     |      0      |   0
  6  |     1     |      1      |   0
  7  |     0     |      1      |   0
  8  |     1     |      0      |   1

PHASE 2: The HALT Signal (Executing 1100 instruction)
  9  |     0     |      0      |   0  <-- Frozen!
 10  |     1     |      0      |   0  <-- Frozen!
 11  |     0     |      0      |   0  <-- Frozen!
 12  |     1     |      0      |   0  <-- Frozen!

PHASE 3: The RESET Button (Hard Reboot)
 13  |     0     |      0      |   0  <-- Cleared!
 14  |     1     |      0      |   0  <-- Cleared!
 15  |     0     |      0      |   0  <-- Cleared!
 16  |     1     |      0      |   0  <-- Cleared!
 17  |     0     |

In [64]:
import sys
sys.path.append("../src")
from utils import int_to_8bit_list

def test_full_timing_integration():
    print("Testing BasicTiming + ControlSignal Integration...\n")
    timing = BasicTiming()
    cu = ControlSignal()

    def clean(sig):
        if isinstance(sig, list): return sig[0]
        return sig

    # ==========================================
    # 1. System Reset
    # ==========================================
    print("--- Booting and Resetting System ---")
    timing(clock=0, reset=1, halt=0)
    cu(cycle_clk=0, pulse=0, reset=1, latch1=[0]*8)
    
    # Release the reset
    timing(clock=0, reset=0, halt=0)
    cu(cycle_clk=0, pulse=0, reset=0, latch1=[0]*8)

    # ==========================================
    # 2. Executing ADD B (Opcode: 0x80)
    # ==========================================
    opcode_ADD_B = 0x80 
    print("\n--- Executing ADD B (0x80) ---")
    
    # We will simulate 16 raw clock ticks (8 full oscillator cycles).
    # ADD B need 2 cycle to fetch, ADD, PC
    # and need 2 execute cycle: 1-Register Array Enable, ALU clock
    #                           2-ALU enable and Accumlate clock
    # This is enough time to get through Fetch 1, Exec 1, and Exec 2.
    for step in range(16):
        # Alternate the raw clock between 0 and 1
        raw_clk = step % 2
        
        # 1. Tick the Timing circuit
        timing(clock=raw_clk, reset=0, halt=0)
        c_clk = timing.read_cycle_clk()
        pls = timing.read_pulse()
        
        # 2. Feed the new timing signals into the Control Unit
        raw_bus = cu(c_clk, pls, reset=0, latch1=int_to_8bit_list(opcode_ADD_B))
        bus = {k: clean(v) for k, v in raw_bus.items()}
        
        # 3. Analyze what the Control Unit is doing
        
        # SETUP PHASE: Cycle Clock is 1, but Pulse hasn't fired yet
        if c_clk == 1 and pls == 0:
            print(f"Raw Tick {step:2d} | SETUP PHASE | cycle_clk=1, pulse=0")
            
            if bus["pc_enable"] == 1 and bus["ram_data_out_enable"] == 1:
                print("    [Fetch 1 Setup] PC and RAM enabled on buses.")
            if bus["inc_enable"] == 1 and bus["ram_data_out_enable"] == 1:
                print("    [Fetch 2 Setup] PC and RAM enabled on buses.")
             
            if bus["ra_enable"] == 1:
                print("    [Exec 1 Setup] Register Array enabled (Reg B on bus).")
            if bus["alu_enable"] == 1:
                print("    [Exec 2 Setup] ALU enabled (Result on bus).")
        
        # PULSE PHASE: Cycle Clock is 1, and Pulse fires to lock data in
        elif c_clk == 1 and pls == 1:
            print(f"Raw Tick {step:2d} | PULSE PHASE | cycle_clk=1, pulse=1")
            
            if bus["inst_latch_1_clk"] == 1 and bus["inc_dec_clk"]:
                print("    [Fetch 1 Pulse] Latch 1 saves Opcode! PC increments!")
            if bus["pc_clk"] == 1:
                print("    [Fetch 2 Pulse] Latch 1 saves Opcode! PC increments!")
            if bus["alu_clk"] == 1:
                print("    [Exec 1 Pulse] ALU saves Register B!")
            if bus["acc_clk"] == 1:
                print("    [Exec 2 Pulse] Accumulator saves Result!")
                
        # DEAD PHASE: Clocks are resting
        else:
            print(f"Raw Tick {step:2d} | RESTING     | cycle_clk={c_clk}, pulse={pls}")

    print("\nTiming & Control Integration PASSED!\n")

if __name__ == "__main__":
    test_full_timing_integration()

Testing BasicTiming + ControlSignal Integration...

--- Booting and Resetting System ---


TypeError: __call__() missing 1 required positional argument: 'enable'

In [ ]:
class CPUSubSet_8080:
    def __init__(self, data_nbits, addr_nbits):
        self.data_nbits = data_nbits
        self.addr_nbits = addr_nbits
        # Central Hardware Components
        self.inst_latch = InstLatch(self.data_nbits)
        self.alu = ALU(self.data_nbits)

        """
        Notice that the B input and the Out output of the ALU are both con-
        nected to the data bus, but the A input is connected directly to the Acc out-
        put of the register array. 
        """
        self.ra = RegisterArray(self.data_nbits)
        self.ram = RAM_64KB(self.data_nbits)
        self.pc = ProgramCounter()
        self.inc_dec = IncrementerDecrementer()

        # Timing and Control
        self.timing = BasicTiming()
        self.control = ControlSignal()

        # System Buses
        self.data_bus = DataBus(num_buffers=5, nbits=self.data_nbits) # RAM, ALU, RA(regs), RA(ACC), InstLatch2 (page 345 and 347)
        self.addr_bus = DataBus(num_buffers=4, nbits=self.addr_nbits) # Inst.Latch2&3, PC, Incr'ter-Decr'ter, RA(HL) (page 353)
        # self.addr_bus = DataBus(num_buffers=3, nbits=self.addr_nbits) # Inst.Latch2&3, PC, Incr'ter-Decr'ter, RA(HL) (page 353)

        self.current_halt_state = 0

    def load_program(self, program_data, start_address=0):
        """helper to load binary instructions into RAM."""
        current_addr = start_address
        for byte_val in program_data:
            addr_bits = int_to_16bit_list(current_addr)
            data_bits = int_to_8bit_list(byte_val)
            self.ram(addr_bits, data_bits, write=1)
            current_addr += 1

    def tick(self, oscillator_clk, external_reset=0):
        # 1. update timing 
        self.timing(oscillator_clk, reset=external_reset, halt=self.current_halt_state)
        cycle_clk = self.timing.read_cycle_clk()
        pulse = self.timing.read_pulse()

        # 2. this signals should always for fetch_cycle_1_out_p370 -> set [ram_data_out_enable, pc_enable] in exec_cycle_1_out_p370 and [inc_dec_clk, instruc_latch_1_clk] in pulse_1
        # signals = self.control(cycle_clk, pulse, external_reset, latch1=None)

        # 2. get current instruction opcode (Opcode is always in Latch 1)
        current_opcode = self.inst_latch.read_latch1()

        # 3. control unit calculates the routing signals
        signals = self.control(cycle_clk=cycle_clk, pulse=pulse, reset=external_reset, latch1=current_opcode)
        # helper to extract 1 or 0 from the signals dictionary securely
        def get_sig(name):
            val = signals[name]
            return val[0] if isinstance(val, list) else val
        
        # handle hardware interrupts (halt)
        self.current_halt_state = get_sig("halt")

        # --------------------------------------------
        # 4. address bus routing
        # --------------------------------------------
        current_address = self.addr_bus([
            self.inst_latch.read_latch2_3(enable_2_3=get_sig("inst_latch2_3_enable")),
            self.pc.readAddr(enable=get_sig("pc_enable")),
            self.inc_dec.readAddr(dec_enable=get_sig("dec_enable"), inc_enable=get_sig("inc_enable")),
            self.ra.read_hl(enable_hl=get_sig("hl_enable"))
        ])

        # --------------------------------------------
        # 5. data bus routing
        # self.data_bus = DataBus(num_buffers=5, nbits=self.data_nbits) # RAM, ALU, RA(regs), RA(ACC), InstLatch2 (page 345 and 347)
        # --------------------------------------------
        current_data = self.data_bus([
            self.ram.read(current_address, enable=get_sig("ram_data_out_enable")),
            self.alu.read_out(enable=get_sig("alu_enable_data_bus_exec_phrase_2_p376")), # acc 
            self.ra.read_register(select=current_opcode[2:5], enable=get_sig("ra_enable")), # [2:5] from page 365
            self.ra.read_accumulator(enable=get_sig("acc_enable_data_bus_exec_phrase_1_p375")),
            self.inst_latch.read_latch2(enable=get_sig("inst_latch_2_enable"))
        ])

        # --------------------------------------------
        # 6. execute writes (clocks and enables capture the bus data)
        # --------------------------------------------
        # memory 
        self.ram(address=current_address, data_in=current_data, write=get_sig("ram_write_enable_data_bus_exec_phrase_1_p375"))

        # Insturction Latches
        self.inst_latch.write_latch1(data=current_data, clock_idx=get_sig("inst_latch_1_clk"))
        self.inst_latch.write_latch2(data=current_data, clock_idx=get_sig("inst_latch_2_clk"))
        self.inst_latch.write_latch3(data=current_data, clock_idx=get_sig("inst_latch_3_clk"))

        # RA 
        self.ra.write_accumulator(data=current_data, clk=get_sig("acc_clk"))
        self.ra(data=current_data, select=current_opcode[5:8], clock=get_sig("ra_clk_data_bus_exec_phrase_1_p375"), 
                hl_select=get_sig("hl_select"), hl_clock=get_sig("hl_clk"), addr=current_address)
        
        # ALU 
        self.alu(input_A=self.ra.read_accumulator(enable=get_sig("acc_enable_data_bus_exec_phrase_1_p375")), input_B=current_data, F2_0=current_opcode[2:5], clock=get_sig("alu_clk_data_bus_exec_phrase_1_p375"))

        # PC & Inc-Dec
        self.inc_dec(addrs=current_address, clock=get_sig("inc_dec_clk"))

        self.pc(addr=current_address, clk=get_sig("pc_clk"))

        # --- Debug Print during the Pulse ---
        if pulse == 1:
            pc_hex = f"{bit_list_to_int(self.pc.readAddr(enable=1), signed=False):04X}"
            op_hex = f"{bit_list_to_int(current_opcode, signed=False):02X}"
            addr_hex = f"{bit_list_to_int(current_address, signed=False):04X}"
            data_hex = f"{bit_list_to_int(current_data, signed=False):02X}"
            print(f"Cycle: {cycle_clk} | PC: {pc_hex} | AddrBus: {addr_hex} | DataBus: {data_hex} | Opcode: {op_hex}")

        

In [31]:
def test_cpu_integration():
    print("Powering on CPUSubSet_8080...")
    cpu = CPUSubSet_8080(data_nbits=8, addr_nbits=16)
    
    # MVI L, 00h (0x2E, 0x00)
    # MVI H, 10h (0x26, 0x10)
    # MVI M, 02h (0x36, 0x02)
    # HLT        (0x76)
    program = [0x2E, 0x00, 0x26, 0x10, 0x36, 0x02, 0x76]
    cpu.load_program(program)
    
    # osc = Oscillator()
    
    # print("Starting Clock...")
    # # Enough ticks to get through 7 bytes
    # for tick in range(250): 
    #     if cpu.current_halt_state == 1:
    #         break
    #     osc_clk = osc.level()
    #     cpu.tick(osc_clk)
    #     osc.tick()
        
    print("\n--- FINAL MEMORY CHECK ---")
    # Verify that the CPU successfully wrote to RAM at address 0x1000!
    test_addr = int_to_16bit_list(0x1000)
    final_val = cpu.ram.read(test_addr, enable=1)
    
    val_int = bit_list_to_int(final_val, False)
    print(f"Data at RAM[0x1000] = {val_int:02X} (Expected: 02)")

test_cpu_integration()

Powering on CPUSubSet_8080...

--- FINAL MEMORY CHECK ---
Data at RAM[0x1000] = 00 (Expected: 02)


In [ ]:
# control_bus = {
#             "pc_enable": program_counter_enable_p372,
#             "pc_clk": program_counter_clk_fetch_phrase_p372,
#             "inc_dec_clk": final_inc_dec_clk,
#             "inc_enable": final_increment_enable,
#             "dec_enable": dec_enable,

#             "inst_latch_1_clk": instru_latch_1_clk_p372,
#             "inst_latch_2_clk": instru_latch_2_clk_p372,
#             "inst_latch_3_clk": instru_latch_3_clk_p372,
#             "inst_latch_2_enable": inst_latch2_enable_data_bus_exec_phrase_1_p375,
#             "inst_latch2_3_enable": inst_latch2_3_enable,

#             "ram_data_out_enable": final_ram_data_out_enable,
#             "ram_write_enable_data_bus_exec_phrase_1_p375": ram_write_enable_data_bus_exec_phrase_1_p375,

#             "ra_enable": ra_enable,
#             "ra_clk_data_bus_exec_phrase_1_p375": ra_clk_data_bus_exec_phrase_1_p375,
#             "hl_enable": hl_enable,
#             "hl_select": hl_select,
#             "hl_clk": hl_clk,
#             "acc_enable_data_bus_exec_phrase_1_p375": acc_enable_data_bus_exec_phrase_1_p375,
#             "acc_clk": final_acc_clk,

#             "alu_enable_data_bus_exec_phrase_2_p376": alu_enable_data_bus_exec_phrase_2_p376,
#             "alu_clk_data_bus_exec_phrase_1_p375": alu_clk_data_bus_exec_phrase_1_p375,

#             "halt": halt,
#             "clear_p370": clear_p370
#         }
#         return control_bus        

In [ ]:
# # TEST SCRIPT
# def test_first_cpu_cycle():
#     cpu = CPUSubSet_8080()
#     osc = Oscillator()
#     # MVI L, 00h,
#     # MVI H, 10h,
#     # MVI M, 02h,
#     # HLT
#     program = [0x2e, 0x00, 0x26, 0x10, 0x36, 0x02, 0x76]
#     cpu.load_program(program_data=program)

#     for _ in range(50):
#         osc_state = osc.level()
#         cpu.tick(osc_state)
#         osc.tick()

# test_first_cpu_cycle()


IndentationError: expected an indented block (2608849099.py, line 2)

In [2]:
# %%writefile test_08_ram64kb.py

# import sys
# sys.path.append("../src")
# from utils import RAM_64KB

def test_ram64kb():
# ==========================================
# 1. EXTREME BOUNDARIES TEST
# ==========================================
# def test_64kb_extreme_boundaries():
    """Tests writing to the absolute edges of the 64KB memory space."""
    ram = RAM_64KB()

    data_zeros = [0] * 8
    # Address 0: 0000 0000 0000 0000
    addr_first = [0]*16
    data_first = [1, 0, 1, 0, 1, 0, 1, 0]
    
    # Address 65535: 1111 1111 1111 1111
    addr_last = [1]*16
    data_last = [0, 1, 0, 1, 0, 1, 0, 1]
    
    # Write to opposite ends of the universe
    ram(addr_first, data_first, write=1)
    ram(addr_last, data_last, write=1)
    
    # Verify both stuck
    assert ram.read(addr_first) == data_first, "Failed to read Address 0x0000"
    assert ram.read(addr_last) == data_last, "Failed to read Address 0xFFFF"
    assert ram.read(addr_first, 0) == data_zeros, "leaked signal when enable = 0"
    assert ram.read(addr_last, 0) == data_zeros, "leaked signal when enable = 0"

    
# ==========================================
# 2. CHIP SELECT CROSSTALK STRESS TEST
# ==========================================
# def test_64kb_chip_crosstalk():
    """
    Writes to 3 different RAM_16_8 chips and verifies the routing logic
    didn't accidentally wake up the wrong chips on the motherboard.
    """
    # ram = RAM_64KB()
    
    # # Address A: Chip 0, internal offset 5
    addr_A = [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,1]
    data_A = [1, 1, 0, 0, 0, 0, 1, 1]
    
    # # Address B: Chip 1, internal offset 5
    addr_B = [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,0,1]
    data_B = [0, 0, 1, 1, 1, 1, 0, 0]
    
    # # Address C: Chip 4095, internal offset 5
    addr_C = [1,1,1,1, 1,1,1,1, 1,1,1,1, 0,1,0,1]
    data_C = [1, 1, 1, 1, 1, 1, 1, 1]
    
    # # Write
    ram(addr_A, data_A, write=1)
    ram(addr_B, data_B, write=1)
    ram(addr_C, data_C, write=1)
    
    # # Read back
    assert ram.read(addr_A) == data_A, "Crosstalk corrupted Chip 0"
    assert ram.read(addr_B) == data_B, "Crosstalk corrupted Chip 1"
    assert ram.read(addr_C) == data_C, "Crosstalk corrupted Chip 4095"

    assert ram.read(addr_A, 0) == data_zeros, "leaked signal when enable = 0"
    assert ram.read(addr_B, 0) == data_zeros, "leaked signal when enable = 0"
    assert ram.read(addr_C, 0) == data_zeros, "leaked signal when enable = 0"


# ==========================================
# 3. GHOST WRITE TEST (64KB SCALE)
# ==========================================
# def test_64kb_ghost_write():
    # ram = RAM_64KB()
    addr = [0,1,0,1, 0,1,0,1, 0,1,0,1, 0,1,0,1] # 0x5555
    data_valid = [0, 0, 0, 0, 1, 1, 1, 1]
    data_ghost = [1, 1, 1, 1, 0, 0, 0, 0]
    
    # # Valid write
    ram(addr, data_valid, write=1)
    
    # # Ghost write (Write Enable = 0)
    ram(addr, data_ghost, write=0)
    
    assert ram.read(addr) == data_valid, "WE=0 shield failed at 64KB scale!"
    assert ram.read(addr, 0) == data_zeros, "leaked signal when enable = 0"

test_ram64kb()

In [3]:
int_to_8bit_list(0x11)

[0, 0, 0, 1, 0, 0, 0, 1]

In [ ]:
import sys
sys.path.append("../src")
from utils import RegisterArray, bit_list_to_int, int_to_8bit_list, int_to_16bit_list

def test_register_array_old_and_fail_test():
    print("Testing RegisterArray...")
    ra = RegisterArray(nbits=8)

    # Helper binary addresses for the decoder (0 to 7)
    select_codes = {
        'B': [0, 0, 0], # 0
        'C': [0, 0, 1], # 1
        'D': [0, 1, 0], # 2
        'E': [0, 1, 1], # 3
        'H': [1, 0, 0], # 4
        'L': [1, 0, 1], # 5
        'A': [1, 1, 1]  # 7
    }

    dummy_addr = [0]*16 # Used when we aren't testing HL

    print("  -> Testing Standard Registers (B)...")
    """
    What Your Test Did
ra(0x00, clock=0): You put 0x00 on the data bus and opened the front door. 0x00 walked into the Waiting Room.

ra(0x11, clock=1): You changed the data to 0x11 at the exact same microsecond you slammed the door shut.

In physical hardware, changing the data and the clock at the exact same time creates a "Race Condition." It is unpredictable. The chip might save 0x00, it might save 0x11, or it might get confused and output random voltage (metastability).

# The data wire goes into the first flip-flop. Evaluated FIRST!
self.rs_flipflop1([..., not_clk, data, preset])

# The clock wire locks the second flip-flop. Evaluated SECOND!
self.rs_flipflop2([clear_p370, ..., preset, not_clk])
    """  
    # 1. Write unique values to B
    ra([0]*8, [0,0,0], clock=0, addr=dummy_addr, hl_select=0, hl_clock=0)
    ra(int_to_8bit_list(0x11), select_codes['B'], clock=1, addr=dummy_addr, hl_select=0, hl_clock=0)
    print(bit_list_to_int(ra.read_register(select_codes['B'], enable=1), False))
    
if __name__ == "__main__":
    test_register_array_old_and_fail_test()

Testing RegisterArray...
  -> Testing Standard Registers (B, C, D, E)...
clock_idx: [0, 0, 0, 0, 0, 0, 0, 0], select: [0, 0, 0], data: [0, 0, 0, 0, 0, 0, 0, 0]
datas: [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]]
idx:0, clk: 0
idx:1, clk: 0
idx:2, clk: 0
idx:3, clk: 0
idx:4, clk: 0
idx:5, clk: 0
idx:7, clk: 0
clock_idx: [1, 0, 0, 0, 0, 0, 0, 0], select: [0, 0, 0], data: [0, 0, 0, 1, 0, 0, 0, 1]
datas: [[0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1]]
idx:0, clk: 1
idx:1, clk: 0
idx:2, clk: 0
idx:3, clk: 0
idx:4, clk: 0
idx:5, clk: 0
idx:7, clk: 0
idx: 0, data: [0, 0, 0, 1, 0, 0, 0, 1]
idx: 1, data: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 2, data: [0, 0, 0, 0, 0, 0, 0, 0]
idx: 3, 

TypeError: read_helper() takes 2 positional arguments but 3 were given

In [23]:
import sys
sys.path.append("../src")
from utils import RegisterArray, bit_list_to_int, int_to_8bit_list, int_to_16bit_list

def test_register_array():
    print("Testing RegisterArray...")
    ra = RegisterArray(nbits=8)

    select_codes = {
        'B': [0, 0, 0], 'C': [0, 0, 1], 'D': [0, 1, 0], 'E': [0, 1, 1],
        'H': [1, 0, 0], 'L': [1, 0, 1], 'A': [1, 1, 1]
    }
    dummy_addr = [0]*16

    # --- THE HARDWARE FIX: A helper function to properly pulse the clock! ---
    def write_reg(val, select):
        data = int_to_8bit_list(val)
        ra(data, select, clock=0, addr=dummy_addr, hl_select=0, hl_clock=0) # 1. Setup Data
        ra(data, select, clock=1, addr=dummy_addr, hl_select=0, hl_clock=0) # 2. Pulse High (Lock)
        ra(data, select, clock=0, addr=dummy_addr, hl_select=0, hl_clock=0) # 3. Pulse Low (Close Door)

    print("  -> Testing Standard Registers (B, C, D, E)...")
    write_reg(0x11, select_codes['B'])
    write_reg(0x22, select_codes['C'])
    write_reg(0x33, select_codes['D'])
    write_reg(0x44, select_codes['E'])

    assert bit_list_to_int(ra.read_register(select_codes['B'], enable=1), False) == 0x11, "Reg B failed"
    assert bit_list_to_int(ra.read_register(select_codes['C'], enable=1), False) == 0x22, "Reg C failed"
    assert bit_list_to_int(ra.read_register(select_codes['D'], enable=1), False) == 0x33, "Reg D failed"
    assert bit_list_to_int(ra.read_register(select_codes['E'], enable=1), False) == 0x44, "Reg E failed"


    print("  -> Testing Accumulator Special Paths...")
    # Write via standard bus, read via dedicated Accumulator path
    write_reg(0x77, select_codes['A'])
    assert bit_list_to_int(ra.read_accumulator(enable=1), False) == 0x77, "Acc read path failed"

    # Write via dedicated Accumulator path (Must properly pulse 0 -> 1 -> 0!)
    ra.write_accumulator(int_to_8bit_list(0x88), clk=0)
    ra.write_accumulator(int_to_8bit_list(0x88), clk=1)
    ra.write_accumulator(int_to_8bit_list(0x88), clk=0)
    
    assert bit_list_to_int(ra.read_register(select_codes['A'], enable=1), False) == 0x88, "Acc write path failed"


    print("  -> Testing HL 16-bit Pointer Logic...")
    test_address = int_to_16bit_list(0xABCD)
    # Properly pulse the HL clock!
    ra(data=[0]*8, select=[0,0,0], clock=0, addr=test_address, hl_select=1, hl_clock=0)
    ra(data=[0]*8, select=[0,0,0], clock=0, addr=test_address, hl_select=1, hl_clock=1)
    ra(data=[0]*8, select=[0,0,0], clock=0, addr=test_address, hl_select=1, hl_clock=0)

    hl_out = ra.read_hl(enable_hl=1)
    assert bit_list_to_int(hl_out, False) == 0xABCD, "HL 16-bit pointer write/read failed"
    assert bit_list_to_int(ra.read_register(select_codes['H'], enable=1), False) == 0xAB, "H register corrupted"
    assert bit_list_to_int(ra.read_register(select_codes['L'], enable=1), False) == 0xCD, "L register corrupted"

    print("RegisterArray PASSED!\n")

if __name__ == "__main__":
    test_register_array()

Testing RegisterArray...
  -> Testing Standard Registers (B, C, D, E)...
clock_idx: [0, 0, 0, 0, 0, 0, 0, 0], select: [0, 0, 0], data: [0, 0, 0, 1, 0, 0, 0, 1]
datas: [[0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1]]
idx:0, clk: 0
idx:1, clk: 0
idx:2, clk: 0
idx:3, clk: 0
idx:4, clk: 0
idx:5, clk: 0
idx:7, clk: 0
clock_idx: [1, 0, 0, 0, 0, 0, 0, 0], select: [0, 0, 0], data: [0, 0, 0, 1, 0, 0, 0, 1]
datas: [[0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 1]]
idx:0, clk: 1
idx:1, clk: 0
idx:2, clk: 0
idx:3, clk: 0
idx:4, clk: 0
idx:5, clk: 0
idx:7, clk: 0
clock_idx: [0, 0, 0, 0, 0, 0, 0, 0], select: [0, 0, 0], data: [0, 0, 0, 1, 0, 0, 0, 1]
datas: [[0, 0, 0, 1, 0, 0, 0, 1], [0, 

In [28]:
import sys
sys.path.append("../src")
from utils import Logic, Add_Subtract, ALU, bit_list_to_int, int_to_8bit_list

def test_logic_unit():
    print("Testing Pure Logic Unit...")
    logic = Logic(nin_data=8, nin_control=3)
    
    A = int_to_8bit_list(204) # 11001100
    B = int_to_8bit_list(170) # 10101010
    
    # 1. AND (100) -> 10001000 (136)
    res_and = logic(A, B, [1, 0, 0])
    assert bit_list_to_int(res_and, False) == 136, f"Logic AND failed"
    
    # 2. XOR (101) -> 01100110 (102)
    res_xor = logic(A, B, [1, 0, 1])
    assert bit_list_to_int(res_xor, False) == 102, f"Logic XOR failed"

    # 3. OR (110) -> 11101110 (238)
    res_or = logic(A, B, [1, 1, 0])
    assert bit_list_to_int(res_or, False) == 238, f"Logic OR failed"
    
    print("Logic Unit PASSED!\n")


def test_add_subtract_unit():
    print("Testing Add_Subtract Unit (With Hardware Borrow Logic)...")
    add_sub = Add_Subtract(nin_data=8, nin_control=2)
    
    A = int_to_8bit_list(100)
    B = int_to_8bit_list(50)
    
    # 1. ADD (00) -> 100 + 50 = 150
    cy, res = add_sub(A, B, F1_0=[0, 0], CY_In=0)
    assert bit_list_to_int(res, False) == 150, "ADD failed"
    assert cy == 0, "ADD Carry out should be 0"

    # 2. ADC (01) -> 100 + 50 + 1 (Carry) = 151
    cy, res = add_sub(A, B, F1_0=[0, 1], CY_In=1)
    assert bit_list_to_int(res, False) == 151, "ADC failed"

    # 3. SUB (10) -> 100 - 50 = 50
    # Because 100 > 50, Intel hardware Borrow (CY) should be 0!
    cy, res = add_sub(A, B, F1_0=[1, 0], CY_In=0)
    assert bit_list_to_int(res, False) == 50, "SUB failed"
    assert cy == 0, "SUB Borrow out should be 0"

    # 4. SBB (11) -> 100 - 50 - 1 (Borrow) = 49
    cy, res = add_sub(A, B, F1_0=[1, 1], CY_In=1)
    assert bit_list_to_int(res, False) == 49, "SBB failed"
    
    print("Add_Subtract Unit PASSED!\n")


def test_full_alu():
    print("Testing Full ALU with Flags and Compare...")
    alu = ALU(nin_data=8, nin_control_signal=3)
    
    def pulse_alu(val_A, val_B, op_code):
        A_bits = int_to_8bit_list(val_A)
        B_bits = int_to_8bit_list(val_B)
        alu(A_bits, B_bits, op_code, clock=0)
        alu(A_bits, B_bits, op_code, clock=1)
        alu(A_bits, B_bits, op_code, clock=0)
        return bit_list_to_int(alu.read_out(enable=1), False), alu.read_flags()

    # 1. Standard ADD (000)
    res, flags = pulse_alu(100, 50, [0, 0, 0])
    assert res == 150, "Full ALU ADD failed"

    # 2. Compare (CMP - 111): A = 100, B = 100
    # Output should remain A (100), but Zero Flag should be 1!
    res, flags = pulse_alu(100, 100, [1, 1, 1])
    assert res == 100, f"CMP should return Accumulator (100), but got {res}"
    assert flags[2] == 0, "CMP did not set Carry Flag!"
    assert flags[1] == 1, "CMP did not set Zero Flag!"

    # 3. Compare (CMP - 111): A = 50, B = 100
    # Output should remain A (50), but it requires a borrow (50-100), so CY=1, Sign=1
    res, flags = pulse_alu(50, 100, [1, 1, 1])
    assert res == 50, f"CMP should return Accumulator (50), but got {res}"
    assert flags[2] == 1, "CMP did not set Carry (Borrow) Flag!"
    assert flags[1] == 0, "CMP, zero flag"
    # sign flag only influenced by acc
    # assert flags[0] == 1, "CMP did not set Sign Flag for negative result!"


    res, flags = pulse_alu(100, 50, [1, 1, 1])
    assert res == 100, f"CMP should return Accumulator (100), but got {res}"
    assert flags[2] == 0, "CMP did not set Carry (Borrow) Flag!"
    assert flags[1] == 0, "CMP, zero flag"
    # sign flag only influenced by acc
    # assert flags[0] == 1, "CMP did not set Sign Flag for negative result!"

    print("Full ALU PASSED!\n")


if __name__ == "__main__":
    # test_logic_unit()
    # test_add_subtract_unit()
    test_full_alu()

Testing Full ALU with Flags and Compare...
Full ALU PASSED!



In [35]:
import sys
sys.path.append("../src")
from utils import Logic, Add_Subtract, ALU, bit_list_to_int, int_to_8bit_list

# ==========================================
# TEST 1: The Pure Logic Unit
# ==========================================
def test_logic_unit():
    print("Testing Pure Logic Unit...")
    logic = Logic(nin_data=8, nin_control=3)
    
    A = int_to_8bit_list(0b11001100) # 204 (Unsigned)
    B = int_to_8bit_list(0b10101010) # 170 (Unsigned)
    
    # AND (100)
    res = logic(A, B, [1, 0, 0])
    assert bit_list_to_int(res, False) == 0b10001000, "Logic AND failed"
    
    # XOR (101)
    res = logic(A, B, [1, 0, 1])
    assert bit_list_to_int(res, False) == 0b01100110, "Logic XOR failed"

    # OR (110)
    res = logic(A, B, [1, 1, 0])
    assert bit_list_to_int(res, False) == 0b11101110, "Logic OR failed"
    
    # Test Dead Wires (e.g., ADD code should output 0s from Logic block)
    res = logic(A, B, [0, 0, 0])
    assert bit_list_to_int(res, False) == 0, "Logic block didn't shut down for Math code!"
    print("  -> Logic Unit PASSED\n")

# ==========================================
# TEST 2: Add/Subtract with Borrow Logic
# ==========================================
def test_add_subtract_unit():
    print("Testing Add_Subtract Unit (Signed & Borrow Edge Cases)...")
    add_sub = Add_Subtract(nin_data=8, nin_control=2)
    
    # Helper to clean up the call
    def run_math(val_a, val_b, f1_0, cy_in=0):
        a_bits = int_to_8bit_list(val_a)
        b_bits = int_to_8bit_list(val_b)
        cy, res = add_sub(a_bits, b_bits, f1_0, cy_in)
        # We read it back as SIGNED to test Two's complement
        return cy, bit_list_to_int(res, signed=True) 

    # 1. ADD (00): 50 + 20 = 70
    cy, res = run_math(50, 20, [0, 0])
    assert res == 70 and cy == 0, "ADD failed"

    # 2. SUB (10): 20 - 50 = -30
    # Because 20 < 50, Intel Borrow (CY) MUST be 1!
    cy, res = run_math(20, 50, [1, 0])
    assert res == -30, f"SUB failed negative math. Got {res}"
    assert cy == 1, "SUB failed to trigger Borrow flag!"

    # 3. SBB (11): Subtract with incoming borrow
    # We are simulating a multi-byte subtraction where the previous byte had to borrow.
    # Math: 20 - 50 - 1 (borrow) = -31
    cy, res = run_math(20, 50, [1, 1], cy_in=1)
    assert res == -31, f"SBB failed. Got {res}"
    assert cy == 1, "SBB failed to trigger next Borrow flag!"

    # 4. Overflow ADD (00): 120 + 10 = 130
    # In 8-bit signed math, max positive is 127. 130 wraps around to a negative number!
    # 130 in binary is 10000010, which Two's Complement reads as -126.
    cy, res = run_math(120, 10, [0, 0])
    assert res == -126, f"ADD signed overflow failed. Got {res}"
    assert cy == 0, "Carry should NOT be 1 here (Signed overflow is different from Unsigned carry!)"

    print("  -> Add_Subtract Unit PASSED\n")

# ==========================================
# TEST 3: Full ALU & The Compare Operation
# ==========================================
def test_full_alu():
    print("Testing Full ALU (Flags & Compare)...")
    alu = ALU(nin_data=8, nin_control_signal=3)
    
    def pulse_alu(val_A, val_B, op_code):
        A_bits = int_to_8bit_list(val_A)
        B_bits = int_to_8bit_list(val_B)
        
        alu(A_bits, B_bits, op_code, clock=0)
        alu(A_bits, B_bits, op_code, clock=1)
        alu(A_bits, B_bits, op_code, clock=0)
        
        out_int = bit_list_to_int(alu.read_out(enable=1), signed=True)
        flags = alu.read_flags()
        return out_int, flags # Flags: [Sign, Zero, Carry]

    # 1. ADD: Negative + Positive = Zero (Testing Zero Flag)
    # -50 + 50 = 0
    res, flags = pulse_alu(-50, 50, [0, 0, 0])
    assert res == 0, "ALU ADD failed"
    assert flags[1] == 1, "Zero flag did not trigger on 0 result!"

    # 2. Logic OR: Testing Sign Flag
    # 0b10000000 (-128) OR 0b00000001 (1) = 0b10000001 (-127)
    res, flags = pulse_alu(-128, 1, [1, 1, 0])
    assert res == -127, "ALU OR failed"
    assert flags[0] == 1, "Sign flag did not trigger on negative logic result!"
    assert flags[2] == 0, "Logic operations must NOT trigger Carry flag!"

    # 3. COMPARE (111): A == B
    # A=50, B=50. Output must remain 50. Zero flag must be 1. Carry must be 0.
    res, flags = pulse_alu(50, 50, [1, 1, 1])
    assert res == 50, f"Compare modified the accumulator! Got {res}"
    assert flags[1] == 1, "Compare failed to set Zero flag for equal values!"
    assert flags[2] == 0, "Compare set Carry on equal values!"

    # 4. COMPARE (111): A < B
    # A=20, B=50. Output must remain 20. Zero flag must be 0. Carry (Borrow) must be 1.
    res, flags = pulse_alu(20, 50, [1, 1, 1])
    assert res == 20, "Compare modified the accumulator!"
    assert flags[1] == 0, "Compare falsely set Zero flag!"
    assert flags[2] == 1, "Compare failed to set Carry (Borrow) when A < B!"
    
    # 5. COMPARE (111): A > B
    # A=50, B=20. Output must remain 50. Zero flag must be 0. Carry (Borrow) must be 0.
    res, flags = pulse_alu(50, 20, [1, 1, 1])
    assert res == 50, "Compare modified the accumulator!"
    assert flags[1] == 0, "Compare falsely set Zero flag!"
    assert flags[2] == 0, "Compare falsely set Carry (Borrow) when A > B!"

    print("  -> Full ALU PASSED\n")


if __name__ == "__main__":
    test_logic_unit()
    test_add_subtract_unit()
    test_full_alu()

Testing Pure Logic Unit...
  -> Logic Unit PASSED

Testing Add_Subtract Unit (Signed & Borrow Edge Cases)...
  -> Add_Subtract Unit PASSED

Testing Full ALU (Flags & Compare)...
  -> Full ALU PASSED



In [31]:
import sys
sys.path.append("../src")
from utils import NBitAdderWithOverflow, int_to_nbit_list, bit_list_to_int

def test_carry_vs_overflow():
    print("Testing Carry vs Overflow Differences...\n")
    adder = NBitAdderWithOverflow(nbits=8)

    def run_case(name, a_val, b_val):
        a_bits = int_to_nbit_list(a_val, 8)
        b_bits = int_to_nbit_list(b_val, 8)
        
        overflow, sum_bits = adder(a_bits, b_bits, last_carry_in=0)
        carry_out = adder.get_carry_out()
        
        print(f"--- {name} ---")
        print(f"  A: {a_val:4d}  ({a_bits})")
        print(f"  B: {b_val:4d}  ({b_bits})")
        print(f"Sum: {bit_list_to_int(sum_bits, signed=True):4d}  ({sum_bits})")
        print(f"Carry Out: {carry_out} | Overflow: {overflow}\n")

    # Case 1: Neither
    # 50 + 20 = 70. Doesn't exceed 255 (No Carry). Logic holds (No Overflow).
    run_case("Case 1: No Carry, No Overflow", 50, 20)

    # Case 2: Overflow, but NO Carry
    # 120 + 10 = 130. Doesn't exceed 255 (No Carry). But Positive + Positive looks Negative! (Overflow).
    run_case("Case 2: No Carry, BUT Overflow!", 120, 10)

    # Case 3: Carry, but NO Overflow
    # -5 + -2 = -7. Exceeds 8 bits (Carry out happens). Logic holds (Neg + Neg = Neg).
    run_case("Case 3: Carry, BUT No Overflow!", -5, -2)

    # Case 4: Both
    # -120 + -10 = -130 (which wraps to +126). Exceeds 8 bits (Carry Out). Neg + Neg looks Positive! (Overflow).
    run_case("Case 4: Carry AND Overflow", -120, -10)

test_carry_vs_overflow()

Testing Carry vs Overflow Differences...

--- Case 1: No Carry, No Overflow ---
  A:   50  ([0, 0, 1, 1, 0, 0, 1, 0])
  B:   20  ([0, 0, 0, 1, 0, 1, 0, 0])
Sum:   70  ([0, 1, 0, 0, 0, 1, 1, 0])
Carry Out: 0 | Overflow: 0

--- Case 2: No Carry, BUT Overflow! ---
  A:  120  ([0, 1, 1, 1, 1, 0, 0, 0])
  B:   10  ([0, 0, 0, 0, 1, 0, 1, 0])
Sum: -126  ([1, 0, 0, 0, 0, 0, 1, 0])
Carry Out: 0 | Overflow: 1

--- Case 3: Carry, BUT No Overflow! ---
  A:   -5  ([1, 1, 1, 1, 1, 0, 1, 1])
  B:   -2  ([1, 1, 1, 1, 1, 1, 1, 0])
Sum:   -7  ([1, 1, 1, 1, 1, 0, 0, 1])
Carry Out: 1 | Overflow: 0

--- Case 4: Carry AND Overflow ---
  A: -120  ([1, 0, 0, 0, 1, 0, 0, 0])
  B:  -10  ([1, 1, 1, 1, 0, 1, 1, 0])
Sum:  126  ([0, 1, 1, 1, 1, 1, 1, 0])
Carry Out: 1 | Overflow: 1



In [38]:
import sys
sys.path.append("../src")
from utils import ALU, bit_list_to_int, int_to_8bit_list

def test_alu_advanced_flags():
    print("Testing ALU Advanced Flag Logic...")
    alu = ALU(nin_data=8, nin_control_signal=3)
    
    def pulse_alu(val_A, val_B, op_code):
        A_bits = int_to_8bit_list(val_A)
        B_bits = int_to_8bit_list(val_B)
        
        alu(A_bits, B_bits, op_code, clock=0)
        alu(A_bits, B_bits, op_code, clock=1)
        alu(A_bits, B_bits, op_code, clock=0)
        
        out_int = bit_list_to_int(alu.read_out(enable=1), signed=True)
        flags = alu.read_flags()
        return out_int, flags # Flags: [Sign, Zero, Carry]

    # ==========================================
    # 1. COMPARE EDGE CASES (Opcode: 111)
    # ==========================================
    print("  -> Testing Compare: A == B")
    res, flags = pulse_alu(50, 50, [1, 1, 1])
    assert res == 50, "Data bus altered during Compare!"
    assert flags == [0, 1, 0], f"Expected S=0, Z=1, CY=0. Got {flags}"

    print("  -> Testing Compare: A < B (Generates Negative & Borrow)")
    res, flags = pulse_alu(50, 100, [1, 1, 1])
    assert res == 50, "Data bus altered during Compare!"
    # 50 - 100 = -50. Sign only influenced by input_A(Acc). Borrow (CY) should be 1. Zero should be 0.
    assert flags == [0, 0, 1], f"Expected S=1, Z=0, CY=1. Got {flags}"

    print("  -> Testing Compare: A > B")
    res, flags = pulse_alu(100, 50, [1, 1, 1])
    assert res == 100, "Data bus altered during Compare!"
    # 100 - 50 = 50. Sign should be 0. Borrow (CY) should be 0. Zero should be 0.
    assert flags == [0, 0, 0], f"Expected S=0, Z=0, CY=0. Got {flags}"

    # ==========================================
    # 2. LOGIC EDGE CASES (Opcodes: 1xx)
    # ==========================================
    print("  -> Testing Logic OR: Generates Negative Result")
    # A = 0b10000000 (-128), B = 0b00000001 (1)
    # OR result = 0b10000001 (-127). 
    res, flags = pulse_alu(-128, 1, [1, 1, 0])
    assert res == -127, f"Logic OR failed. Got {res}"
    # Result is negative (S=1), not zero (Z=0). Logic operations clear_p370 carry (CY=0).
    assert flags == [1, 0, 0], f"Expected S=1, Z=0, CY=0. Got {flags}"

    print("  -> Testing Logic XOR: Generates Zero Result & Clears Carry")
    # To prove logic clears the carry flag, we first force the Carry flag to 1 using an addition overflow:
    pulse_alu(200, 200, [0, 0, 0]) 
    # now sign, zero and carry is [1, 0, 1]

    # Now execute XOR on identical inputs to get 0
    res, flags = pulse_alu(15, 15, [1, 0, 1])
    assert res == 0, f"Logic XOR failed. Got {res}"
    # Result is zero (Z=1). Sign is 0. Old Carry MUST be wiped out to 0!
    assert flags == [0, 1, 0], f"Expected S=0, Z=1, CY=0. Got {flags}"

    print("Advanced ALU Flags PASSED!\n")

if __name__ == "__main__":
    test_alu_advanced_flags()

Testing ALU Advanced Flag Logic...
  -> Testing Compare: A == B
  -> Testing Compare: A < B (Generates Negative & Borrow)
  -> Testing Compare: A > B
  -> Testing Logic OR: Generates Negative Result
  -> Testing Logic XOR: Generates Zero Result & Clears Carry
Advanced ALU Flags PASSED!



In [39]:
bits = int_to_8bit_list(200)
print(bits)
bits = int_to_8bit_list(-50)
print(bits)

[1, 1, 0, 0, 1, 0, 0, 0]
[1, 1, 0, 0, 1, 1, 1, 0]


In [40]:
import sys
sys.path.append("../src")
from utils import ProgramCounter, bit_list_to_int, int_to_16bit_list

def test_program_counter():
    print("Testing Program Counter...")
    pc = ProgramCounter()

    # ==========================================
    # 1. TEST RESET FUNCTION
    # ==========================================
    print("  -> Testing Reset...")
    pc.Reset()
    out = pc.readAddr(enable=1)
    assert bit_list_to_int(out, signed=False) == 0x0000, f"PC failed to reset to 0. Got {out}"

    # ==========================================
    # 2. TEST CLOCK & SETUP TIME (Writing Data)
    # ==========================================
    print("  -> Testing Clock Pulse (Rising Edge)...")
    test_addr = int_to_16bit_list(0x1234)
    
    # Step A: Setup time (Clock is 0, door is open but data not locked in vault)
    pc(test_addr, clk=0)
    # The output should technically still be 0 here because it hasn't locked!
    # Note: EdgeTriggered flips on the rising edge, so the vault hasn't changed yet.
    assert bit_list_to_int(pc.readAddr(enable=1), False) == 0x0000, "PC prematurely changed before clock pulse!"

    # Step B: Rising Edge (Clock goes 1, door shuts, locks 0x1234 in vault)
    pc(test_addr, clk=1)
    assert bit_list_to_int(pc.readAddr(enable=1), False) == 0x1234, "PC failed to latch data on rising edge!"

    # Step C: Falling Edge (Clock goes 0, ready for next time)
    pc(test_addr, clk=0)
    assert bit_list_to_int(pc.readAddr(enable=1), False) == 0x1234, "PC lost data on falling edge!"

    # ==========================================
    # 3. TEST TRI-STATE BUFFER (Enable Signal)
    # ==========================================
    print("  -> Testing Output Enable (Tri-State Buffer)...")
    # Disable the output. The bus should read all 0s (dead wires), NOT 0x1234.
    out_disabled = pc.readAddr(enable=0)
    assert bit_list_to_int(out_disabled, signed=False) == 0x0000, "Tri-State leak! PC output data while disabled."
    
    # Re-enable the output. Should see 0x1234 again.
    out_enabled = pc.readAddr(enable=1)
    assert bit_list_to_int(out_enabled, signed=False) == 0x1234, "PC lost data while disabled!"

    # ==========================================
    # 4. TEST DIRECT SETTERS (SetMaxAddr / SetAddr)
    # ==========================================
    print("  -> Testing Direct Setters (Asynchronous Preset/Clear)...")
    
    # Test SetMaxAddr (Should force to 0xFFFF)
    pc.SetMaxAddr()
    assert bit_list_to_int(pc.readAddr(enable=1), False) == 0xFFFF, "SetMaxAddr failed to set all bits to 1!"

    # Test SetAddr (Should force to arbitrary value without needing a clock pulse)
    custom_addr = int_to_16bit_list(0xABCD)
    pc.SetAddr(custom_addr)
    assert bit_list_to_int(pc.readAddr(enable=1), False) == 0xABCD, "SetAddr failed to force custom address!"

    print("Program Counter PASSED!\n")

if __name__ == "__main__":
    test_program_counter()

Testing Program Counter...
  -> Testing Reset...
  -> Testing Clock Pulse (Rising Edge)...
  -> Testing Output Enable (Tri-State Buffer)...
  -> Testing Direct Setters (Asynchronous Preset/Clear)...
Program Counter PASSED!



In [41]:
import sys
sys.path.append("../src")
from utils import InstLatch, bit_list_to_int, int_to_8bit_list

def test_inst_latch():
    print("Testing Instruction Latch...")
    latch = InstLatch(nbits=8)

    # Helper to safely pulse the clock (Setup Time -> Rising Edge -> Falling Edge)
    def write_to_latch(target_latch_num, val):
        data_bits = int_to_8bit_list(val)
        if target_latch_num == 1:
            latch.write_latch1(data_bits, 0)
            latch.write_latch1(data_bits, 1)
            latch.write_latch1(data_bits, 0)
        elif target_latch_num == 2:
            latch.write_latch2(data_bits, 0)
            latch.write_latch2(data_bits, 1)
            latch.write_latch2(data_bits, 0)
        elif target_latch_num == 3:
            latch.write_latch3(data_bits, 0)
            latch.write_latch3(data_bits, 1)
            latch.write_latch3(data_bits, 0)

    # ==========================================
    # 1. Test Opcode Latch (Latch 1)
    # ==========================================
    print("  -> Testing Latch 1 (Opcode)...")
    # Latch 1 has no Tri-State buffer. It is always feeding the Control Unit.
    write_to_latch(1, 0x3E) # Opcode for MVI A
    out1 = latch.read_latch1()
    assert bit_list_to_int(out1, False) == 0x3E, "Latch 1 failed to hold Opcode"

    # ==========================================
    # 2. Test Immediate Data Latch (Latch 2)
    # ==========================================
    print("  -> Testing Latch 2 (Immediate Data & Tri-State)...")
    write_to_latch(2, 0x45) # Random data byte
    
    # Check disabled state (Should output 0s to not crash the Data Bus)
    out2_disabled = latch.read_latch2(enable=0)
    assert bit_list_to_int(out2_disabled, False) == 0x00, "Latch 2 leaked data to Data Bus when disabled!"
    
    # Check enabled state
    out2_enabled = latch.read_latch2(enable=1)
    assert bit_list_to_int(out2_enabled, False) == 0x45, "Latch 2 failed to output to Data Bus!"

    # ==========================================
    # 3. Test Address Latches (Latches 2 & 3 combined)
    # ==========================================
    print("  -> Testing Latches 2 & 3 (16-bit Address Bus)...")
    # Let's say we are executing LDA 0xABCD. 
    # Because of Little Endian: Byte 2 is 0xCD, Byte 3 is 0xAB.
    write_to_latch(2, 0xCD)
    write_to_latch(3, 0xAB)

    # Check disabled state
    out23_disabled = latch.read_latch2_3(enable_2_3=0)
    assert bit_list_to_int(out23_disabled, False) == 0x0000, "Latches 2&3 leaked data to Address Bus!"

    # Check enabled state
    out23_enabled = latch.read_latch2_3(enable_2_3=1)
    result_16bit = bit_list_to_int(out23_enabled, False)
    
    assert result_16bit == 0xABCD, f"Combined Address failed! Expected 0xABCD, got 0x{result_16bit:04X}. (Did you fix the endianness bug?)"

    print("Instruction Latch PASSED!\n")

if __name__ == "__main__":
    test_inst_latch()

Testing Instruction Latch...
  -> Testing Latch 1 (Opcode)...
  -> Testing Latch 2 (Immediate Data & Tri-State)...
  -> Testing Latches 2 & 3 (16-bit Address Bus)...
Instruction Latch PASSED!



In [42]:
import sys
sys.path.append("../src")
from utils import IncrementerDecrementer, bit_list_to_int, int_to_16bit_list

def test_incrementer_decrementer():
    print("Testing Incrementer/Decrementer...")
    inc_dec = IncrementerDecrementer()

    # Helper to properly simulate the clock pulsing
    def pulse_clock(val):
        addrs = int_to_16bit_list(val)
        inc_dec(addrs, clock=0)
        inc_dec(addrs, clock=1)
        inc_dec(addrs, clock=0)

    # ==========================================
    # 1. Test Incrementing
    # ==========================================
    print("  -> Testing Standard Increment...")
    pulse_clock(0x1000)
    out_inc = inc_dec.readAddr(dec_enable=0, inc_enable=1)
    assert bit_list_to_int(out_inc, False) == 0x1001, "Increment failed!"

    print("  -> Testing Increment Rollover (0xFFFF + 1)...")
    pulse_clock(0xFFFF)
    out_inc_roll = inc_dec.readAddr(dec_enable=0, inc_enable=1)
    assert bit_list_to_int(out_inc_roll, False) == 0x0000, "Increment Rollover failed!"

    # ==========================================
    # 2. Test Decrementing
    # ==========================================
    print("  -> Testing Standard Decrement...")
    pulse_clock(0x2000)
    out_dec = inc_dec.readAddr(dec_enable=1, inc_enable=0)
    assert bit_list_to_int(out_dec, False) == 0x1FFF, "Decrement failed!"

    print("  -> Testing Decrement Underflow (0x0000 - 1)...")
    pulse_clock(0x0000)
    out_dec_under = inc_dec.readAddr(dec_enable=1, inc_enable=0)
    assert bit_list_to_int(out_dec_under, False) == 0xFFFF, "Decrement Underflow failed!"

    # ==========================================
    # 3. Test Tri-State Buffer Isolation
    # ==========================================
    print("  -> Testing Tri-State Output Isolation...")
    pulse_clock(0xABCD)
    # Read with BOTH enables set to 0
    out_disabled = inc_dec.readAddr(dec_enable=0, inc_enable=0)
    assert bit_list_to_int(out_disabled, False) == 0x0000, "Tri-State leak! Output data while disabled."

    print("Incrementer/Decrementer PASSED!\n")

if __name__ == "__main__":
    test_incrementer_decrementer()

Testing Incrementer/Decrementer...
  -> Testing Standard Increment...
  -> Testing Increment Rollover (0xFFFF + 1)...
  -> Testing Standard Decrement...
  -> Testing Decrement Underflow (0x0000 - 1)...
  -> Testing Tri-State Output Isolation...
Incrementer/Decrementer PASSED!

